# Practice Lab: Linear Regression (One Variable)
### Predicting Employee Salary from Years of Experience

Welcome! This notebook is a self-contained practice lab on **univariate linear regression** — same core concept as the classic "restaurant profit vs. city population" lab, but rebuilt from scratch with a brand-new dataset and no external files. Everything needed to run this notebook top-to-bottom is included right here, so you can upload it straight to GitHub as your own worked example.

## Outline
- [1 - Packages](#1)
- [2 - Problem Statement](#2)
- [3 - Dataset](#3)
  - [3.1 View the variables](#3.1)
  - [3.2 Check the dimensions](#3.2)
  - [3.3 Visualize the data](#3.3)
- [4 - Refresher on Linear Regression](#4)
- [5 - Compute Cost](#5)
  - [Exercise 1: `compute_cost`](#ex01)
- [6 - Gradient Descent](#6)
  - [Exercise 2: `compute_gradient`](#ex02)
  - [Exercise 3: `gradient_descent`](#ex03)
- [7 - Learning the Parameters](#7)
- [8 - Plotting the Fit](#8)
- [9 - Making Predictions](#9)
- [10 - (Bonus) Comparing with `scikit-learn`](#10)


<a name="1"></a>
## 1 - Packages

We only need two libraries for the core lab:
- [numpy](https://www.numpy.org) — for arrays and numerical operations.
- [matplotlib](https://matplotlib.org) — for plotting.

(A `scikit-learn` check is used only in the optional bonus section at the very end.)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import math

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid') if 'seaborn-v0_8-whitegrid' in plt.style.available else None

<a name="2"></a>
## 2 - Problem Statement

Suppose you work in the HR analytics team of a growing tech company.

- The company wants a quick, transparent way to estimate a **fair starting salary** for a candidate based on their **years of professional experience**.
- You have historical data: for a sample of current employees, you know their years of experience and their current annual salary.
- Your job: fit a straight line through this data so that, given a *new* candidate's years of experience, you can predict a reasonable salary.

This is exactly the same shape of problem as "predict profit from population" — one input feature, one numeric output, and a straight-line relationship — just a different real-world story.


<a name="3"></a>
## 3 - Dataset

Instead of loading a `.txt` file, we'll **generate** a synthetic-but-realistic dataset right here in the notebook. That keeps this notebook fully self-contained (no extra files to upload alongside it), while still behaving like real, noisy, real-world data.

- `x_train`: years of experience (0 to ~12 years)
- `y_train`: annual salary, in units of **$1,000** (so `55.0` means \$55,000)

We use a fixed random seed so your results exactly match the expected outputs below.


In [ ]:
# Generate the dataset
np.random.seed(42)

m = 60  # number of training examples (employees)

# Years of experience: random values between 0 and 12
x_train = np.round(np.random.uniform(0, 12, m), 2)
x_train = np.sort(x_train)

# True underlying relationship: salary = base + slope * experience + noise
true_base = 35      # starting salary with 0 experience ($35,000)
true_slope = 6.5     # ~$6,500 raise per year of experience
noise = np.random.normal(0, 6, m)  # some realistic noise

y_train = true_base + true_slope * x_train + noise
y_train = np.round(y_train, 2)

<a name="3.1"></a>
#### 3.1 View the variables

Let's inspect the data, just like you would with any new dataset.


In [ ]:
# print x_train
print("Type of x_train:", type(x_train))
print("First five elements of x_train (years of experience):\n", x_train[:5])

In [ ]:
# print y_train
print("Type of y_train:", type(y_train))
print("First five elements of y_train (salary in $1000s):\n", y_train[:5])

`x_train` holds years of experience for each employee, and `y_train` holds their salary in thousands of dollars — e.g., `41.5` means an annual salary of \$41,500.


<a name="3.2"></a>
#### 3.2 Check the dimensions

It's good practice to confirm the shape of your data before doing anything else.


In [ ]:
print('The shape of x_train is:', x_train.shape)
print('The shape of y_train is:', y_train.shape)
print('Number of training examples (m):', len(x_train))

**Expected Output**:
```
The shape of x_train is: (60,)
The shape of y_train is: (60,)
Number of training examples (m): 60
```


<a name="3.3"></a>
#### 3.3 Visualize the data

A scatter plot is a great first look at a two-variable relationship.


In [ ]:
plt.scatter(x_train, y_train, marker='x', c='r')
plt.title("Salary vs. Years of Experience")
plt.ylabel('Salary (in $1,000s)')
plt.xlabel('Years of Experience')
plt.show()

<a name="4"></a>
## 4 - Refresher on Linear Regression

In linear regression, you fit a straight line to your data:

$$ f_{w,b}(x^{(i)}) = wx^{(i)} + b \tag{1}$$

- $w$ and $b$ are the **parameters** of the model — $w$ is the slope, $b$ is the intercept.
- You want to find the $w, b$ that make $f_{w,b}(x^{(i)})$ as close as possible to $y^{(i)}$ for every training example $i$.

To measure "how close," we use the **squared error cost function**:

$$ J(w,b) = \frac{1}{2m} \sum\limits_{i=0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2 \tag{2}$$

You will:
1. Implement `compute_cost` to measure how good a given $(w, b)$ is.
2. Implement `compute_gradient` to find which direction to adjust $(w, b)$ to reduce the cost.
3. Implement `gradient_descent`, which repeatedly applies the gradient to learn the best $(w, b)$.


<a name="5"></a>
## 5 - Compute Cost

<a name="ex01"></a>
### Exercise 1: `compute_cost`

Complete the `compute_cost` function below. Recall:

$$ J(w,b) = \frac{1}{2m} \sum\limits_{i=0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2 $$

where $f_{w,b}(x^{(i)}) = wx^{(i)} + b$.

**Steps**:
1. Loop over all `m` training examples (or use vectorized numpy operations).
2. For each example, compute the prediction $f_{w,b}(x^{(i)})$.
3. Compute the squared error $(f_{w,b}(x^{(i)}) - y^{(i)})^2$.
4. Accumulate the total, then divide by $2m$.


In [ ]:
def compute_cost(x, y, w, b):
    '''
    Computes the cost function for linear regression.

    Args:
        x (ndarray): Shape (m,) Input to the model (years of experience)
        y (ndarray): Shape (m,) Target values (salary in $1000s)
        w, b (scalar): Parameters of the model

    Returns:
        total_cost (float): The cost of using w, b as parameters
                             for linear regression to fit the data points in x and y
    '''
    m = x.shape[0]

    total_cost = 0
    cost_sum = 0

    for i in range(m):
        f_wb = w * x[i] + b
        cost = (f_wb - y[i]) ** 2
        cost_sum += cost

    total_cost = (1 / (2 * m)) * cost_sum

    return total_cost

Now let's test `compute_cost` with some initial values for $w$ and $b$.


In [ ]:
# Compute cost with some initial values for parameters w, b
initial_w = 2
initial_b = 1

cost = compute_cost(x_train, y_train, initial_w, initial_b)
print(type(cost))
print(f'Cost at initial w: {cost:.3f}')

# Quick sanity check: cost should be a non-negative float
assert cost >= 0, "Cost should never be negative!"
print("Sanity check passed \u2705")

<a name="6"></a>
## 6 - Gradient Descent

The gradient descent algorithm is:

$$\begin{align*}& \text{repeat until convergence:} \; \lbrace \newline \; & \phantom {0000} b := b -  \alpha \frac{\partial J(w,b)}{\partial b} \newline       \; & \phantom {0000} w := w -  \alpha \frac{\partial J(w,b)}{\partial w} \tag{3}  \; & \newline & \rbrace\end{align*}$$

where, parameters $w, b$ are updated *simultaneously*, $\alpha$ is the learning rate, and:

$$ \frac{\partial J(w,b)}{\partial b}  = \frac{1}{m} \sum\limits_{i=0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)}) \tag{4}$$
$$ \frac{\partial J(w,b)}{\partial w}  = \frac{1}{m} \sum\limits_{i=0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})x^{(i)} \tag{5}$$

<a name="ex02"></a>
### Exercise 2: `compute_gradient`

Complete `compute_gradient` to return $\frac{\partial J(w,b)}{\partial w}$ and $\frac{\partial J(w,b)}{\partial b}$.


In [ ]:
def compute_gradient(x, y, w, b):
    '''
    Computes the gradient for linear regression.

    Args:
        x (ndarray): Shape (m,) Input to the model (years of experience)
        y (ndarray): Shape (m,) Target values (salary in $1000s)
        w, b (scalar): Parameters of the model

    Returns:
        dj_dw (scalar): The gradient of the cost w.r.t. the parameter w
        dj_db (scalar): The gradient of the cost w.r.t. the parameter b
    '''
    m = x.shape[0]

    dj_dw = 0
    dj_db = 0

    for i in range(m):
        f_wb = w * x[i] + b
        dj_dw_i = (f_wb - y[i]) * x[i]
        dj_db_i = (f_wb - y[i])
        dj_dw += dj_dw_i
        dj_db += dj_db_i

    dj_dw = dj_dw / m
    dj_db = dj_db / m

    return dj_dw, dj_db

Let's check the gradient at the initial parameters.


In [ ]:
initial_w = 0
initial_b = 0

tmp_dj_dw, tmp_dj_db = compute_gradient(x_train, y_train, initial_w, initial_b)
print('Gradient at initial w, b (zeros):', tmp_dj_dw, tmp_dj_db)

<a name="ex03"></a>
### Exercise 3: `gradient_descent`

Now let's put it together. `gradient_descent` repeatedly calls `compute_gradient` and `compute_cost` to update $w$ and $b$ and track how the cost decreases over iterations.


In [ ]:
def gradient_descent(x, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters):
    '''
    Performs batch gradient descent to learn w, b. Updates w, b by taking
    num_iters gradient steps with learning rate alpha.

    Args:
        x (ndarray):  Shape (m,)
        y (ndarray):  Shape (m,)
        w_in, b_in (scalar): initial values of parameters
        cost_function: function to compute cost
        gradient_function: function to compute the gradient
        alpha (float): Learning rate
        num_iters (int): number of iterations to run gradient descent

    Returns:
        w (scalar): Updated value of parameter after running gradient descent
        b (scalar): Updated value of parameter after running gradient descent
        J_history (list): History of cost values
        p_history (list): History of parameters [w,b]
    '''
    m = len(x)

    J_history = []
    p_history = []
    b = b_in
    w = w_in

    for i in range(num_iters):
        dj_dw, dj_db = gradient_function(x, y, w, b)

        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        if i < 100000:
            cost = cost_function(x, y, w, b)
            J_history.append(cost)
            p_history.append([w, b])

        if i % math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:5}: Cost {J_history[-1]:8.2f}   "
                  f"w: {w: .4f}, b: {b: .4f}")

    return w, b, J_history, p_history

<a name="7"></a>
## 7 - Learning the Parameters

Now let's actually run gradient descent on our salary dataset.

**Note:** Since years of experience ranges from 0-12 (a modest range) and salaries are in the tens, a learning rate around `0.01` works well here without needing feature scaling.


In [ ]:
# initialize parameters
initial_w = 0.0
initial_b = 0.0

# gradient descent settings
iterations = 1500
alpha = 0.01

w, b, J_history, _ = gradient_descent(
    x_train, y_train, initial_w, initial_b,
    compute_cost, compute_gradient, alpha, iterations
)

print("\nLearned parameters: w = {:.4f}, b = {:.4f}".format(w, b))

**Expected behavior**: the cost should steadily decrease each iteration, and the learned `w` should land close to the true underlying slope (~6.5) and `b` close to the true base salary (~35), since that's what we used to generate the data.


In [ ]:
# Plot the cost vs. iteration to confirm gradient descent is converging
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(J_history)
ax1.set_title("Cost vs. iteration (start)")
ax1.set_ylabel('Cost')
ax1.set_xlabel('Iteration step')

ax2.plot(1000 + np.arange(len(J_history[1000:])), J_history[1000:])
ax2.set_title("Cost vs. iteration (end)")
ax2.set_ylabel('Cost')
ax2.set_xlabel('Iteration step')
plt.tight_layout()
plt.show()

<a name="8"></a>
## 8 - Plotting the Fit

Let's overlay our learned line on top of the training data.


In [ ]:
predicted = w * x_train + b

plt.plot(x_train, predicted, c="b", label="Linear fit")
plt.scatter(x_train, y_train, marker='x', c='r', label="Training data")
plt.title("Salary vs. Years of Experience")
plt.ylabel('Salary (in $1,000s)')
plt.xlabel('Years of Experience')
plt.legend()
plt.show()

<a name="9"></a>
## 9 - Making Predictions

With our learned $w, b$, we can predict the salary for a new candidate given their years of experience.

Let's predict the salary for someone with **3 years** and **10 years** of experience.


In [ ]:
experience_1 = 3
predicted_salary_1 = w * experience_1 + b
print(f'For {experience_1} years of experience, we predict a salary of ${predicted_salary_1 * 1000:,.2f}')

experience_2 = 10
predicted_salary_2 = w * experience_2 + b
print(f'For {experience_2} years of experience, we predict a salary of ${predicted_salary_2 * 1000:,.2f}')

Feel free to try other values of `experience_1` / `experience_2` above and re-run the cell to see how the prediction changes!


<a name="10"></a>
## 10 - (Bonus) Comparing with `scikit-learn`

As a sanity check, let's see how close our from-scratch implementation is to a production-grade library. This cell is optional — if `scikit-learn` isn't installed, just skip it or run `!pip install scikit-learn`.


In [ ]:
try:
    from sklearn.linear_model import LinearRegression

    model = LinearRegression()
    model.fit(x_train.reshape(-1, 1), y_train)

    print("Our gradient descent   -> w: {:.4f}, b: {:.4f}".format(w, b))
    print("scikit-learn (closed form) -> w: {:.4f}, b: {:.4f}".format(
        model.coef_[0], model.intercept_))
except ImportError:
    print("scikit-learn not installed — skipping this optional comparison.\n"
          "You can install it with: !pip install scikit-learn")

## Congratulations!

You've implemented linear regression with one variable completely from scratch, on a brand-new dataset:
- Built and visualized a dataset (salary vs. years of experience)
- Implemented the cost function `compute_cost`
- Implemented the gradient function `compute_gradient`
- Implemented `gradient_descent` and used it to learn the best-fit line
- Used the learned model to make new predictions

This is the same underlying algorithm used in the "profit vs. population" lab — just applied to a new problem, which is great practice for really understanding what's going on under the hood rather than just re-running someone else's cells.

**Next step idea:** try changing `true_slope`, `true_base`, the noise level, or `m` in Section 3 and re-run the whole notebook to see how the fit adapts.
